# 保存与加载 (Serialization)
- torch.save(tensor, 'path.pt'): 保存张量到磁盘。
- torch.load('path.pt'): 从磁盘加载张量。

| 特性 | torch.save | torch.load |
| :--- | :--- | :--- |
| 作用 | 序列化（保存） | 反序列化（加载） |
| 常用参数 | `obj` (对象), `f` (文件路径) | `f` (文件路径), `map_location` (设备映射) |
| 推荐格式 | `.pt` 或 `.pth` | `.pt` 或 `.pth` |
| 注意事项 | 建议保存前将 Tensor 移至 CPU (`tensor.cpu()`) 以提升跨平台兼容性 | 加载时需关注 `map_location` 避免设备冲突；关注 `weights_only` 确保安全 |


torch.save 和 torch.load 是 PyTorch 中用于持久化存储数据的核心工具。<font color='red'>它们基于 Python 的 pickle 模块实现序列化，能够完整保留张量的数值、数据类型（dtype）、形状（shape）以及设备信息。</font>

简单来说，它们的作用是将内存中的 Tensor“冻结”成文件，以便稍后读取或在不同机器间传输。

## 💾 torch.save(tensor, 'path.pt'): 保存张量

这个函数用于将张量（或其他 Python 对象）序列化并保存到磁盘文件中。

- 核心功能：将 Tensor 对象转换为二进制流并写入文件。
- 文件扩展名：虽然你可以使用任何扩展名，但 PyTorch 社区约定俗成使用 .pt 或 .pth。
- 底层机制：<font color='red'>默认使用 Python 的 pickle 模块。这意味着它不仅能保存数据，还能保存对象的元数据。</font>

In [ ]:
import torch

# 1. 创建一个张量 (例如在 GPU 上)
if torch.cuda.is_available():
    tensor = torch.randn(3, 4).cuda()
else:
    tensor = torch.randn(3, 4)

# 2. 保存到磁盘
# 推荐做法：如果张量在 GPU 上，保存前最好转为 CPU，以保证兼容性
torch.save(tensor.cpu(), 'my_tensor.pt') 

## 📂 torch.load('path.pt'): 加载张量

这个函数用于读取磁盘上的文件，并将其反序列化为 PyTorch 张量。

- 核心功能：<font color='red'>从文件中读取二进制数据</font>，重建 Tensor 对象。
- 关键参数<font color='red'> map_location</font>：这是加载时最重要的参数，用于指定将数据加载到哪个设备（CPU 或 GPU）。

### ⚠️ 常见的“坑”与最佳实践

在使用这两个函数时，新手常遇到以下两个问题，请务必注意：

1. 设备不匹配问题 (Device Mismatch)

如果你在一个有 GPU 的机器上保存了模型（例如保存到了 cuda:0），然后在没有 GPU 的机器（或仅使用 CPU 的环境）上加载，直接运行 torch.load 会报错，因为它试图将张量加载到不存在的 GPU 上。
- 解决方案：使用 map_location 强制指定加载设备。

In [ ]:
# 强制加载到 CPU，无论原文件是在哪里保存的
loaded_tensor = torch.load('my_tensor.pt', map_location=torch.device('cpu'))

2. 安全性问题 (Security Warning)

由于 torch.load 底层使用 pickle，而 pickle 在加载文件时可以执行任意代码。如果你加载了一个来自不可信来源（如陌生人上传的文件）的 .pt 文件，可能会导致恶意代码在你的机器上运行。

- 解决方案：<font color='red'>加载不可信文件时，务必设置 weights_only=True。这会限制加载的内容仅为张量和基本数据类型，阻止恶意代码执行。</font>

In [ ]:
# 安全模式加载
safe_tensor = torch.load('untrusted_file.pt', weights_only=True)

# pickle

pickle 是 Python 的标准库模块，被誉为 Python 对象序列化的“瑞士军刀”。它的核心作用是将内存中复杂的 Python 对象（如列表、字典、甚至自定义类的实例）转换为字节流，以便存储到磁盘或通过网络传输，并在需要时精确还原。

你可以把它想象成一个“打包机”：
- Pickling (腌制/序列化)：将对象打包成字节流。
- Unpickling (解腌/反序列化)：将字节流拆包还原为对象。

##  核心功能与用法
pickle 提供了四个核心函数，分为“文件操作”和“内存操作”两组：

| 函数 | 方向 | 描述 | 适用场景 |
| :--- | :--- | :--- | :--- |
| `pickle.dump(obj, file)` | 序列化 | 将对象写入文件 | 保存模型、缓存数据到磁盘 |
| `pickle.load(file)` | 反序列化 | 从文件读取并还原对象 | 加载之前保存的数据 |
| `pickle.dumps(obj)` | 序列化 | 将对象转换为字节串 (bytes) | 网络传输、数据库存储 (返回 bytes) |
| `pickle.loads(bytes)` | 反序列化 | 从字节串还原对象 | 接收网络数据包 (接收 bytes) |

注意：操作文件时，必须使用二进制模式 ('wb' 写入, 'rb' 读取)。



In [1]:
import pickle

# 1. 准备一个复杂对象 (包含字典、列表、元组)
data = {
    "name": "Alice",
    "scores": [95, 88, 76],
    "info": (25, "Engineer")
}

# 2. 序列化 (保存到文件)
# 使用 'wb' 模式写入二进制文件
with open('data.pkl', 'wb') as f:
    pickle.dump(data, f)

# 3. 反序列化 (从文件加载)
# 使用 'rb' 模式读取二进制文件
with open('data.pkl', 'rb') as f:
    loaded_data = pickle.load(f)

print(loaded_data) 
# 输出: {'name': 'Alice', 'scores': [95, 88, 76], 'info': (25, 'Engineer')}

{'name': 'Alice', 'scores': [95, 88, 76], 'info': (25, 'Engineer')}


##  为什么使用 Pickle？(vs JSON)

虽然 JSON 也很流行，但 pickle 在 Python 生态中有其独特的优势：

- <font color='red'>支持几乎所有 Python 对象</font>：JSON 只能处理基本数据类型（字符串、数字、列表、字典）。而 pickle 可以处理自定义类实例、函数、闭包，甚至复杂的递归结构。
- 二进制格式：<font color='red'>pickle 生成的是二进制数据，通常比 JSON 的文本格式更紧凑，读写速度也更快。</font>
- <font color='red'>无需手动转换</font>：你不需要像处理 JSON 那样编写 to_json 或 from_json 的转换逻辑，pickle 能自动保留对象的引用关系和类型信息。

| 特性 | Pickle | JSON |
| :--- | :--- | :--- |
| 数据格式 | 二进制 (人类不可读) | 文本 (人类可读) |
| 支持类型 | 几乎所有 Python 对象 | 仅基础类型 (dict, list, str, int...) |
| 跨语言 | Python 专用 | 通用 (几乎所有语言都支持) |
| 安全性 | 低 (存在代码执行风险) | 高 |



## ⚠️ 致命弱点：安全性警告

这是使用 pickle 时必须时刻牢记的第一原则：

<font color='red'>绝对不要反序列化来自不可信来源的数据！</font>
- 原因：<font color='red'>pickle 的反序列化过程本质上等同于执行任意 Python 代码。</font>
如果黑客构造了一个恶意的 .pkl 文件，当你调用 pickle.load() 时，它可能会在你的电脑上执行删除文件、窃取密码或安装病毒等恶意操作。
- 安全建议：只加载你自己生成的文件，或者你完全信任的源生成的文件。<font color='red'>如果需要与外部系统交换数据，请优先使用 JSON 或专门的序列化格式（如 Protobuf）。</font>

## 进阶：协议版本与 PyTorch 的关系
### 协议版本 (Protocol Versions)

pickle 有多个版本（0-5），版本越高，序列化效率越高，支持的类型也越新。
- 兼容性：高版本 Python 生成的 pickle 文件，低版本 Python 可能无法读取。
- 建议：在保存文件时，通常不需要手动指定，使用默认值即可。但在跨 Python 版本传输时需注意。

### 与 PyTorch 的关系

你可能在 PyTorch 中经常用到它。torch.save() 和 torch.load() 的底层其实就是封装了 pickle。
- 当你保存一个神经网络模型时，PyTorch 使用 pickle 将模型的权重和结构序列化。
- 这也是为什么加载 PyTorch 模型时，如果模型类的定义代码不存在，加载会失败的原因（因为 pickle 需要导入原始类来重建对象）。